# 13 — Cross-Attention Multimodal Drug Response

Each modality is projected to 256-d via a dedicated encoder, then fused with attention.

**Fixed across all variants:**
- RNA → `OmicsEncoder` → 256-d
- Protein → `OmicsEncoder` → 256-d
- Omics fusion: `CrossAttentionBlock(query=rna, key/value=protein)` → 256-d

**Variation axes:**

| | Drug FP encoder | Drug Graph GNN |
|---|---|---|
| **Drug via cross-attention** | Variant AX | Variant BX |
| **Drug via concat** | Variant AY | Variant BY |

Drug integration cross-attention direction: `query=drug, key/value=omics_fused`
(drug queries the cell's biological context — see nb13 header for biological rationale).

All variants share identical encoder and attention block implementations.
Each variant gets its own dataloader, instantiation, training, predict, and eval cells.

## Environment setup (Colab or local)

In [1]:
from pathlib import Path

try:
    import google.colab  # noqa: F401
    IN_COLAB = True
except ImportError:
    IN_COLAB = False

if IN_COLAB:
    !pip install -q torch-geometric rdkit
    from google.colab import drive
    drive.mount('/content/drive')
    BASE_PATH = Path('/content/drive/MyDrive/multiomics_project')
else:
    BASE_PATH = Path('..')

print(f"Running on {'Colab' if IN_COLAB else 'local'} | BASE_PATH = {BASE_PATH}")

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
Running on Colab | BASE_PATH = /content/drive/MyDrive/multiomics_project


## GPU check

In [2]:
import torch

DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print('Device:', DEVICE)
if DEVICE.type == 'cpu':
    print('WARNING: no GPU detected — Runtime > Change runtime type > GPU')

Device: cuda


## Imports and config

In [3]:
import json
import warnings
warnings.filterwarnings('ignore')

import numpy as np
import pandas as pd
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import DataLoader, Dataset, TensorDataset
from torch_geometric.data import Batch, Data
from torch_geometric.loader import DataLoader as GeoDataLoader
from torch_geometric.nn import GCNConv, global_mean_pool
from sklearn.model_selection import GroupShuffleSplit
from scipy.stats import pearsonr, spearmanr, linregress
from sklearn.metrics import r2_score, roc_auc_score
from rdkit import Chem
from rdkit.Chem import rdFingerprintGenerator
from tqdm.notebook import tqdm
import matplotlib.pyplot as plt

In [4]:
DATA_DIR       = BASE_PATH / 'data' / 'GDSC2'
PROCESSED_DIR  = BASE_PATH / 'data' / 'processed'
SPLITS_DIR     = BASE_PATH / 'data' / 'splits'
RESULTS_DIR    = BASE_PATH / 'results' / 'cross_attention'
RESULTS_DIR.mkdir(parents=True, exist_ok=True)
(RESULTS_DIR / 'checkpoints').mkdir(exist_ok=True)

COL_CELL_LINE   = 'cell_line_name'
COL_DRUG        = 'drug_name'
COL_IC50        = 'LN_IC50'
COL_CELLOSAURUS = 'cellosaurus_id'
COL_TISSUE      = 'tissue'

EMBED_DIM        = 256
TOP_K_FEATURES   = 1000
VALIDATION_RATIO = 0.1
RANDOM_STATE     = 42
BATCH_SIZE       = 64
NUM_EPOCHS       = 20
PATIENCE         = 10
DROPOUT          = 0.3
MODALITY_DROPOUT = 0.3   # probability of zeroing protein embedding during training

QUICK_TEST = True

## Load response pairs and splits

In [5]:
pairs = pd.read_parquet(PROCESSED_DIR / 'response_pairs.parquet')

with open(SPLITS_DIR / 'splits.json') as f:
    folds = json.load(f)

with open(SPLITS_DIR / 'holdout_groups.json') as f:
    holdout_groups = json.load(f)

print(f'{len(pairs):,} pairs loaded')
for fd in folds:
    print(f"fold {fd['fold']}: train={len(fd['train']):,} | "
          f"lco={len(fd['lco_test']):,} | ldo={len(fd['ldo_test']):,} | "
          f"lto={len(fd['lto_test']):,} | lpo={len(fd['lpo_test']):,}")

176,197 pairs loaded
fold 0: train=107,421 | lco=17,470 | ldo=18,404 | lto=25,964 | lpo=20,613
fold 1: train=118,444 | lco=17,331 | ldo=18,515 | lto=13,579 | lpo=16,475
fold 2: train=97,262 | lco=18,173 | ldo=18,126 | lto=35,198 | lpo=23,832
fold 3: train=111,375 | lco=17,849 | ldo=18,140 | lto=21,451 | lpo=19,147
fold 4: train=103,126 | lco=17,762 | ldo=18,533 | lto=28,869 | lpo=21,721


## Load omics features and drug inputs

In [6]:
rna         = pd.read_csv(DATA_DIR / 'gene_expression.csv',  index_col=0)
protein     = pd.read_csv(DATA_DIR / 'proteomics.csv',        index_col=0)
drug_smiles = pd.read_csv(DATA_DIR / 'drug_smiles.csv')

# Deduplicate — duplicate cellosaurus_id rows cause silent row-count mismatches
rna     = rna[~rna.index.duplicated(keep='first')].iloc[:, 1:]
protein = protein[~protein.index.duplicated(keep='first')].fillna(0)

OMICS = {'rna': rna, 'protein': protein}

print(f'RNA:     {rna.shape[1]:,} genes    x {rna.shape[0]:,} cell lines')
print(f'Protein: {protein.shape[1]:,} proteins x {protein.shape[0]:,} cell lines')
print(f'Drugs:   {drug_smiles[COL_DRUG].nunique():,}')

RNA:     17,737 genes    x 1,010 cell lines
Protein: 6,692 proteins x 860 cell lines
Drugs:   246


In [7]:
def build_drug_fingerprints(drug_smiles_df: pd.DataFrame,
                             radius: int = 2, n_bits: int = 2048) -> dict:
    generator = rdFingerprintGenerator.GetMorganGenerator(radius=radius, fpSize=n_bits)
    fps = {}
    for _, row in drug_smiles_df.iterrows():
        mol = Chem.MolFromSmiles(row['canonical_smiles'])
        if mol is None:
            continue
        fps[row[COL_DRUG]] = np.array(generator.GetFingerprint(mol), dtype=np.float32)
    return fps


def atom_to_features(atom) -> list:
    return [
        atom.GetAtomicNum(), atom.GetDegree(), atom.GetFormalCharge(),
        int(atom.GetHybridization()), int(atom.GetIsAromatic()), atom.GetTotalNumHs(),
    ]


def smiles_to_graph(smiles: str):
    mol = Chem.MolFromSmiles(smiles)
    if mol is None:
        return None
    x = torch.tensor([atom_to_features(a) for a in mol.GetAtoms()], dtype=torch.float)
    edges = []
    for bond in mol.GetBonds():
        i, j = bond.GetBeginAtomIdx(), bond.GetEndAtomIdx()
        edges += [[i, j], [j, i]]
    edge_index = torch.tensor(edges, dtype=torch.long).t().contiguous()
    return Data(x=x, edge_index=edge_index)


def build_drug_graphs(drug_smiles_df: pd.DataFrame) -> dict:
    graphs = {}
    for _, row in drug_smiles_df.iterrows():
        g = smiles_to_graph(row['canonical_smiles'])
        if g is not None:
            graphs[row[COL_DRUG]] = g
    return graphs


drug_fp     = build_drug_fingerprints(drug_smiles)
drug_graphs = build_drug_graphs(drug_smiles)
print(f'Fingerprints: {len(drug_fp):,} | Graphs: {len(drug_graphs):,}')

Fingerprints: 246 | Graphs: 246


## Train / validation split

In [8]:
def make_validation_indices(train_idx: np.ndarray,
                             fraction: float = VALIDATION_RATIO,
                             seed: int = RANDOM_STATE):
    sub = pairs.loc[train_idx]

    def axis_holdout(group_values: pd.Series, seed_offset: int) -> set:
        gss = GroupShuffleSplit(n_splits=1, test_size=fraction,
                                random_state=seed + seed_offset)
        _, val_idx = next(gss.split(np.arange(len(group_values)),
                                    groups=group_values))
        return set(group_values.iloc[val_idx])

    cell_ho   = axis_holdout(sub[COL_CELLOSAURUS], 0)
    drug_ho   = axis_holdout(sub[COL_DRUG],        1)
    tissue_ho = axis_holdout(sub[COL_TISSUE],      2)

    is_val = (
        sub[COL_CELLOSAURUS].isin(cell_ho)
        | sub[COL_DRUG].isin(drug_ho)
        | sub[COL_TISSUE].isin(tissue_ho)
    ).to_numpy()

    return train_idx[~is_val], train_idx[is_val]

train_inner_idx, val_idx = make_validation_indices(np.array(folds[0]['train']))
print(f'train_inner: {len(train_inner_idx):,} | val: {len(val_idx):,}')

train_inner: 79,187 | val: 28,234


## Feature selection

In [9]:
def select_top_variance_omics(arm: str, train_idx: np.ndarray, k: int) -> pd.Index:
    train_cells = pairs.loc[train_idx, COL_CELLOSAURUS].unique()
    compact = OMICS[arm].loc[OMICS[arm].index.intersection(train_cells)]
    return compact.var(axis=0).sort_values(ascending=False).index[:k]

top_cols = {
    'rna':     select_top_variance_omics('rna',     np.array(folds[0]['train']), TOP_K_FEATURES),
    'protein': select_top_variance_omics('protein', np.array(folds[0]['train']), TOP_K_FEATURES),
}
print(f"RNA: {len(top_cols['rna']):,} | Protein: {len(top_cols['protein']):,}")

RNA: 1,000 | Protein: 1,000


## Feature matrices

FP variants (AX, AY) use TensorDataset. Graph variants (BX, BY) use a custom Dataset — built below.

In [10]:
def build_feature_matrix(idx: np.ndarray, top_cols: dict) -> dict:
    """Returns dict with keys 'rna', 'protein', 'drug_fp', 'y'."""
    sub   = pairs.loc[idx]
    cells = sub[COL_CELLOSAURUS]

    rna_X     = OMICS['rna'].loc[cells, top_cols['rna']].to_numpy().astype(np.float32)
    protein_X = OMICS['protein'].loc[cells, top_cols['protein']].to_numpy().astype(np.float32)
    drug_X    = np.vstack([drug_fp[d] for d in sub[COL_DRUG]]).astype(np.float32)
    y         = sub[COL_IC50].to_numpy().astype(np.float32)

    assert not np.isnan(rna_X).any(),     'NaNs in RNA'
    assert not np.isnan(protein_X).any(), 'NaNs in protein'
    assert not np.isnan(y).any(),         'NaNs in target'

    return {'rna': rna_X, 'protein': protein_X, 'drug_fp': drug_X, 'y': y}


def make_fp_dataloader(matrices: dict, batch_size: int, shuffle: bool) -> DataLoader:
    """TensorDataset for FP variants: (rna, protein, drug_fp, y)."""
    dataset = TensorDataset(
        torch.from_numpy(matrices['rna']),
        torch.from_numpy(matrices['protein']),
        torch.from_numpy(matrices['drug_fp']),
        torch.from_numpy(matrices['y']),
    )
    return DataLoader(dataset, batch_size=batch_size, shuffle=shuffle, drop_last=shuffle)

In [11]:
print('Building feature matrices...')
train_matrices = build_feature_matrix(train_inner_idx, top_cols)
val_matrices   = build_feature_matrix(val_idx,          top_cols)
lco_matrices   = build_feature_matrix(np.array(folds[0]['lco_test']), top_cols)
ldo_matrices   = build_feature_matrix(np.array(folds[0]['ldo_test']), top_cols)
lto_matrices   = build_feature_matrix(np.array(folds[0]['lto_test']), top_cols)
lpo_matrices   = build_feature_matrix(np.array(folds[0]['lpo_test']), top_cols)
print('Done.')
for name, m in [('train', train_matrices), ('val', val_matrices), ('lco', lco_matrices)]:
    print(f"  {name}: rna={m['rna'].shape}, protein={m['protein'].shape}, "
          f"drug_fp={m['drug_fp'].shape}, y={m['y'].shape}")

Building feature matrices...
Done.
  train: rna=(79187, 1000), protein=(79187, 1000), drug_fp=(79187, 2048), y=(79187,)
  val: rna=(28234, 1000), protein=(28234, 1000), drug_fp=(28234, 2048), y=(28234,)
  lco: rna=(17470, 1000), protein=(17470, 1000), drug_fp=(17470, 2048), y=(17470,)


## Graph dataset (BX, BY variants)

In [12]:
class OmicsGraphDataset(Dataset):
    """Yields (rna, protein, drug_graph, y) for graph-drug variants.
    Rows whose drug has no valid graph are dropped at construction time.
    """

    def __init__(self, idx: np.ndarray, top_cols: dict):
        sub = pairs.loc[idx]
        has_graph = sub[COL_DRUG].isin(drug_graphs)
        n_dropped = (~has_graph).sum()
        if n_dropped:
            print(f'Dropping {n_dropped} rows with no valid drug graph')
        self.sub = sub[has_graph].reset_index(drop=True)

        cells = self.sub[COL_CELLOSAURUS]
        self.rna_X     = OMICS['rna'].loc[cells, top_cols['rna']].to_numpy().astype(np.float32)
        self.protein_X = OMICS['protein'].loc[cells, top_cols['protein']].to_numpy().astype(np.float32)
        self.y         = self.sub[COL_IC50].to_numpy().astype(np.float32)

    def __len__(self):
        return len(self.sub)

    def __getitem__(self, i):
        g = drug_graphs[self.sub.loc[i, COL_DRUG]]
        return (
            torch.from_numpy(self.rna_X[i]),
            torch.from_numpy(self.protein_X[i]),
            Data(x=g.x, edge_index=g.edge_index),
            torch.tensor(self.y[i], dtype=torch.float),
        )


def make_graph_dataloader(idx: np.ndarray, top_cols: dict,
                           batch_size: int, shuffle: bool) -> GeoDataLoader:
    return GeoDataLoader(
        OmicsGraphDataset(idx, top_cols),
        batch_size=batch_size, shuffle=shuffle, drop_last=shuffle,
    )

## Model architecture

```
RNA    → OmicsEncoder   → rna_emb   (256-d)
Protein → OmicsEncoder  → prot_emb  (256-d)
CrossAttentionBlock(query=rna_emb, key/value=prot_emb) → omics_fused (256-d)

Drug FP  → DrugMLPEncoder → drug_emb (256-d)   [variants AX, AY]
Drug Graph → DrugGNN      → drug_emb (256-d)   [variants BX, BY]

Drug integration:
  AX / BX: CrossAttentionBlock(query=drug_emb, key/value=omics_fused) → fused (256-d)
  AY / BY: concat(omics_fused, drug_emb) → Linear(512, 256) → fused (256-d)

Predictor: fused (256-d) → Linear(256,128) → ReLU → Linear(128,1)
```

**Biological directionality of drug integration attention:**
`query=drug` means the drug embedding asks 'which aspects of this cell biology are
relevant to my mechanism of action?' — the cell's omics profile is the context being
queried, not the other way around.

In [13]:
class OmicsEncoder(nn.Module):
    """Projects one omics modality to EMBED_DIM."""

    def __init__(self, input_dim: int, hidden: int = 512,
                 out_dim: int = EMBED_DIM, dropout_prob: float = DROPOUT):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(input_dim, hidden),
            nn.BatchNorm1d(hidden),
            nn.Dropout(dropout_prob),
            nn.ReLU(),
            nn.Linear(hidden, out_dim),
            nn.BatchNorm1d(out_dim),
            nn.ReLU(),
        )

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        return self.net(x)

In [14]:
class DrugMLPEncoder(nn.Module):
    """Projects drug fingerprint (2048-d) to EMBED_DIM."""

    def __init__(self, input_dim: int = 2048, hidden: int = 512,
                 out_dim: int = EMBED_DIM, dropout_prob: float = DROPOUT):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(input_dim, hidden),
            nn.BatchNorm1d(hidden),
            nn.Dropout(dropout_prob),
            nn.ReLU(),
            nn.Linear(hidden, out_dim),
            nn.BatchNorm1d(out_dim),
            nn.ReLU(),
        )

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        return self.net(x)

In [15]:
class DrugGNN(nn.Module):
    """3-layer GCNConv + global mean pool → EMBED_DIM.
    Same architecture as nb09 DrugGNN, head removed (embedding only).
    """

    def __init__(self, in_dim: int = 6, hidden: int = 256,
                 out_dim: int = EMBED_DIM, dropout_prob: float = DROPOUT):
        super().__init__()
        self.conv1 = GCNConv(in_dim, hidden)
        self.conv2 = GCNConv(hidden, hidden)
        self.conv3 = GCNConv(hidden, out_dim)
        self.bn1   = nn.BatchNorm1d(hidden)
        self.bn2   = nn.BatchNorm1d(hidden)
        self.drop  = nn.Dropout(dropout_prob)

    def forward(self, x, edge_index, batch):
        x = F.relu(self.bn1(self.conv1(x, edge_index)))
        x = self.drop(x)
        x = F.relu(self.bn2(self.conv2(x, edge_index)))
        x = self.drop(x)
        x = self.conv3(x, edge_index)
        return global_mean_pool(x, batch)  # (batch_size, out_dim)

In [16]:
class CrossAttentionBlock(nn.Module):
    """Single cross-attention block with residual connection and FFN.

    query attends to key/value.
    Input and output dim are both `dim`.
    """

    def __init__(self, dim: int = EMBED_DIM, n_heads: int = 4,
                 dropout_prob: float = DROPOUT):
        super().__init__()
        self.attn  = nn.MultiheadAttention(dim, n_heads, dropout=dropout_prob,
                                            batch_first=True)
        self.norm1 = nn.LayerNorm(dim)
        self.norm2 = nn.LayerNorm(dim)
        self.ffn   = nn.Sequential(
            nn.Linear(dim, dim * 2),
            nn.ReLU(),
            nn.Linear(dim * 2, dim),
        )

    def forward(self, query: torch.Tensor, context: torch.Tensor) -> torch.Tensor:
        """query, context: (batch, dim) — reshaped internally to (batch, 1, dim)."""
        q = query.unsqueeze(1)
        k = context.unsqueeze(1)
        attn_out, _ = self.attn(q, k, k)               # (batch, 1, dim)
        x = self.norm1(q + attn_out).squeeze(1)        # residual + norm
        x = self.norm2(x + self.ffn(x))               # FFN + norm
        return x                                        # (batch, dim)

In [17]:
class CrossAttentionDRP(nn.Module):
    """Cross-attention drug response predictor.

    drug_modality  : 'fp' or 'graph'
    drug_integration: 'attn' or 'concat'
    modality_dropout: probability of zeroing protein embedding at train time
    """

    def __init__(self,
                 rna_dim: int,
                 protein_dim: int,
                 drug_modality: str = 'fp',
                 drug_integration: str = 'attn',
                 drug_fp_dim: int = 2048,
                 embed_dim: int = EMBED_DIM,
                 modality_dropout: float = MODALITY_DROPOUT,
                 dropout_prob: float = DROPOUT):
        super().__init__()
        assert drug_modality    in ('fp', 'graph')
        assert drug_integration in ('attn', 'concat')
        self.drug_modality    = drug_modality
        self.drug_integration = drug_integration
        self.modality_dropout = modality_dropout

        # Omics encoders
        self.rna_encoder     = OmicsEncoder(rna_dim,     dropout_prob=dropout_prob)
        self.protein_encoder = OmicsEncoder(protein_dim, dropout_prob=dropout_prob)

        # Omics cross-attention: RNA queries protein
        self.omics_attn = CrossAttentionBlock(embed_dim, dropout_prob=dropout_prob)

        # Drug encoder
        if drug_modality == 'fp':
            self.drug_encoder = DrugMLPEncoder(drug_fp_dim, dropout_prob=dropout_prob)
        else:
            self.drug_encoder = DrugGNN(dropout_prob=dropout_prob)

        # Drug integration
        if drug_integration == 'attn':
            # Drug queries omics_fused
            self.drug_attn = CrossAttentionBlock(embed_dim, dropout_prob=dropout_prob)
            predictor_dim  = embed_dim
        else:
            # concat(omics_fused, drug_emb) -> project back to embed_dim
            self.drug_proj = nn.Sequential(
                nn.Linear(embed_dim * 2, embed_dim),
                nn.BatchNorm1d(embed_dim),
                nn.ReLU(),
            )
            predictor_dim  = embed_dim

        self.predictor = nn.Sequential(
            nn.Linear(predictor_dim, 128),
            nn.ReLU(),
            nn.Dropout(dropout_prob),
            nn.Linear(128, 1),
        )

    def forward(self, rna, protein, drug_input, drug_batch=None):
        """drug_input: FP tensor (batch, 2048) for 'fp', or (x, edge_index) for 'graph'.
        drug_batch: PyG batch vector, required for 'graph' modality.
        """
        # Encode omics
        rna_emb  = self.rna_encoder(rna)
        prot_emb = self.protein_encoder(protein)

        # Protein modality dropout during training
        if self.training and self.modality_dropout > 0:
            mask     = (torch.rand(prot_emb.shape[0], device=prot_emb.device)
                        > self.modality_dropout).float().unsqueeze(1)
            prot_emb = prot_emb * mask

        # Omics fusion: RNA queries protein
        omics_fused = self.omics_attn(query=rna_emb, context=prot_emb)

        # Encode drug
        if self.drug_modality == 'fp':
            drug_emb = self.drug_encoder(drug_input)
        else:
            x, edge_index = drug_input
            drug_emb = self.drug_encoder(x, edge_index, drug_batch)

        # Drug integration
        if self.drug_integration == 'attn':
            # Drug queries the cell's omics context
            fused = self.drug_attn(query=drug_emb, context=omics_fused)
        else:
            fused = self.drug_proj(torch.cat([omics_fused, drug_emb], dim=1))

        return self.predictor(fused).squeeze(-1)


def count_parameters(model: nn.Module) -> int:
    return sum(p.numel() for p in model.parameters() if p.requires_grad)

## Training loops

In [18]:
def fit_fp_variant(model: nn.Module,
                   train_loader: DataLoader,
                   val_loader: DataLoader,
                   variant_name: str,
                   num_epochs: int,
                   patience: int,
                   checkpoint_path: Path,
                   device: torch.device):
    """Training loop for FP variants (AX, AY).
    Batch unpacking: (rna, protein, drug_fp, y).
    """
    criterion = nn.HuberLoss()
    optimizer = torch.optim.Adam(model.parameters(), lr=1e-4, weight_decay=1e-4)

    history = {'train_loss': [], 'train_cor': [], 'val_loss': [], 'val_cor': []}
    best_val_loss = float('inf')
    best_weights  = None
    wait          = 0

    print(f'Training variant: {variant_name} | params: {count_parameters(model):,}')

    for epoch in range(num_epochs):
        for phase in ['val', 'train']:
            loader       = train_loader if phase == 'train' else val_loader
            phase_device = device if phase == 'train' else torch.device('cpu')
            model.to(phase_device)
            model.train() if phase == 'train' else model.eval()

            batch_losses, preds, targets = [], [], []

            for rna, protein, drug_fp_t, y in tqdm(
                    loader, desc=f'epoch {epoch+1} {phase}', leave=False):
                rna, protein, drug_fp_t, y = (
                    rna.to(phase_device), protein.to(phase_device),
                    drug_fp_t.to(phase_device), y.to(phase_device),
                )

                if phase == 'train':
                    optimizer.zero_grad()
                    pred = model(rna, protein, drug_fp_t)
                    loss = criterion(pred, y)
                    loss.backward()
                    optimizer.step()
                else:
                    with torch.no_grad():
                        pred = model(rna, protein, drug_fp_t)
                        loss = criterion(pred, y)

                batch_losses.append(loss.item())
                preds.append(pred.detach())
                targets.append(y.detach())

            epoch_loss = sum(batch_losses) / len(batch_losses)
            epoch_cor, _ = pearsonr(
                torch.cat(targets).cpu().numpy(),
                torch.cat(preds).cpu().numpy(),
            )
            history[f'{phase}_loss'].append(epoch_loss)
            history[f'{phase}_cor'].append(epoch_cor)

            if phase == 'val':
                print(f'  epoch {epoch+1:>3} | val loss={epoch_loss:.4f}  pearson_r={epoch_cor:.4f}')
                if epoch_loss < best_val_loss:
                    best_val_loss = epoch_loss
                    best_weights  = {k: v.cpu().clone() for k, v in model.state_dict().items()}
                    wait = 0
                else:
                    wait += 1
                    if wait >= patience:
                        print(f'  Early stopping at epoch {epoch+1}')
                        model.load_state_dict(best_weights)
                        torch.save(best_weights, checkpoint_path)
                        print(f'  Best val loss: {best_val_loss:.4f} -> {checkpoint_path}')
                        return model, history

    model.load_state_dict(best_weights)
    torch.save(best_weights, checkpoint_path)
    print(f'  Best val loss: {best_val_loss:.4f} -> {checkpoint_path}')
    return model, history

In [19]:
def fit_graph_variant(model: nn.Module,
                      train_loader: GeoDataLoader,
                      val_loader: GeoDataLoader,
                      variant_name: str,
                      num_epochs: int,
                      patience: int,
                      checkpoint_path: Path,
                      device: torch.device):
    """Training loop for graph variants (BX, BY).
    PyG DataLoader yields a Batch object; rna and protein are stored as
    regular attributes on each Data item and stacked by PyG's collate.
    """
    criterion = nn.HuberLoss()
    optimizer = torch.optim.Adam(model.parameters(), lr=1e-4, weight_decay=1e-4)

    history = {'train_loss': [], 'train_cor': [], 'val_loss': [], 'val_cor': []}
    best_val_loss = float('inf')
    best_weights  = None
    wait          = 0

    print(f'Training variant: {variant_name} | params: {count_parameters(model):,}')

    for epoch in range(num_epochs):
        for phase in ['val', 'train']:
            loader       = train_loader if phase == 'train' else val_loader
            phase_device = device if phase == 'train' else torch.device('cpu')
            model.to(phase_device)
            model.train() if phase == 'train' else model.eval()

            batch_losses, preds, targets = [], [], []

            for rna, protein, drug_data, y in tqdm(
                    loader, desc=f'epoch {epoch+1} {phase}', leave=False):
                rna       = rna.to(phase_device)
                protein   = protein.to(phase_device)
                drug_data = drug_data.to(phase_device)
                y         = y.to(phase_device)

                if phase == 'train':
                    optimizer.zero_grad()
                    pred = model(rna, protein,
                                 (drug_data.x, drug_data.edge_index),
                                 drug_batch=drug_data.batch)
                    loss = criterion(pred, y)
                    loss.backward()
                    optimizer.step()
                else:
                    with torch.no_grad():
                        pred = model(rna, protein,
                                     (drug_data.x, drug_data.edge_index),
                                     drug_batch=drug_data.batch)
                        loss = criterion(pred, y)

                batch_losses.append(loss.item())
                preds.append(pred.detach())
                targets.append(y.detach())

            epoch_loss = sum(batch_losses) / len(batch_losses)
            epoch_cor, _ = pearsonr(
                torch.cat(targets).cpu().numpy(),
                torch.cat(preds).cpu().numpy(),
            )
            history[f'{phase}_loss'].append(epoch_loss)
            history[f'{phase}_cor'].append(epoch_cor)

            if phase == 'val':
                print(f'  epoch {epoch+1:>3} | val loss={epoch_loss:.4f}  pearson_r={epoch_cor:.4f}')
                if epoch_loss < best_val_loss:
                    best_val_loss = epoch_loss
                    best_weights  = {k: v.cpu().clone() for k, v in model.state_dict().items()}
                    wait = 0
                else:
                    wait += 1
                    if wait >= patience:
                        print(f'  Early stopping at epoch {epoch+1}')
                        model.load_state_dict(best_weights)
                        torch.save(best_weights, checkpoint_path)
                        print(f'  Best val loss: {best_val_loss:.4f} -> {checkpoint_path}')
                        return model, history

    model.load_state_dict(best_weights)
    torch.save(best_weights, checkpoint_path)
    print(f'  Best val loss: {best_val_loss:.4f} -> {checkpoint_path}')
    return model, history

## Prediction and evaluation helpers

In [20]:
def predict_fp_variant(model: nn.Module, matrices: dict,
                        batch_size: int = 512, device: str = 'cpu') -> np.ndarray:
    model.to(device)
    model.eval()
    rna_t     = torch.from_numpy(matrices['rna'])
    protein_t = torch.from_numpy(matrices['protein'])
    drug_t    = torch.from_numpy(matrices['drug_fp'])
    n = rna_t.shape[0]
    preds = []
    with torch.no_grad():
        for i in range(0, n, batch_size):
            pred = model(
                rna_t[i:i+batch_size].to(device),
                protein_t[i:i+batch_size].to(device),
                drug_t[i:i+batch_size].to(device),
            )
            preds.append(pred.cpu())
    return torch.cat(preds).numpy()


def predict_graph_variant(model: nn.Module, loader: GeoDataLoader,
                           device: str = 'cpu') -> tuple:
    """Returns (predictions, targets) — targets come from the loader."""
    model.to(device)
    model.eval()
    preds, targets = [], []
    with torch.no_grad():
        for rna, protein, drug_data, y in loader:
            rna       = rna.to(device)
            protein   = protein.to(device)
            drug_data = drug_data.to(device)
            pred = model(rna, protein,
                         (drug_data.x, drug_data.edge_index),
                         drug_batch=drug_data.batch)
            preds.append(pred.cpu())
            targets.append(y.cpu())
    return torch.cat(preds).numpy(), torch.cat(targets).numpy()

In [21]:
def evaluate_split(y_true: np.ndarray, y_pred: np.ndarray) -> dict:
    corr,  _  = pearsonr(y_true, y_pred)
    spear, _  = spearmanr(y_true, y_pred)
    rmse      = float(np.sqrt(np.mean((y_true - y_pred) ** 2)))
    r2        = float(r2_score(y_true, y_pred))
    slope, _, _, _, std_err = linregress(y_pred, y_true)
    binary = (y_true < np.median(y_true)).astype(int)
    roc    = roc_auc_score(binary, -y_pred) if len(np.unique(binary)) == 2 else np.nan
    return {
        'Pearson r':  round(corr,    4),
        'Spearman r': round(spear,   4),
        'RMSE':       round(rmse,    4),
        'R2':         round(r2,      4),
        'ROC-AUC':    round(roc, 4) if not np.isnan(roc) else roc,
        'Slope':      round(slope,   4),
        'Std err':    round(std_err, 4),
    }


def evaluate_all_splits(variant_name: str, preds: dict) -> pd.DataFrame:
    """preds: dict of split_name -> (y_true, y_pred).
    For 'LPO', the full array is evaluated as 'LPO - all', then each
    lpo_masks sub-split is evaluated separately.
    """
    rows = []
    for split_name, (y_true, y_pred) in preds.items():
        if split_name == 'LPO':
            row = {'variant': variant_name, 'split': 'LPO - all'}
            row.update(evaluate_split(y_true, y_pred))
            rows.append(row)
            for mask_name, mask in lpo_masks.items():
                if mask.sum() == 0:
                    continue
                row = {'variant': variant_name, 'split': mask_name}
                row.update(evaluate_split(y_true[mask], y_pred[mask]))
                rows.append(row)
        else:
            row = {'variant': variant_name, 'split': split_name}
            row.update(evaluate_split(y_true, y_pred))
            rows.append(row)
    return pd.DataFrame(rows)


def build_lpo_masks(fold_idx: int = 0) -> dict:
    cell_ho = set(holdout_groups[fold_idx]['cell_lines_held_out'])
    drug_ho = set(holdout_groups[fold_idx]['drugs_held_out'])
    lpo_idx = np.array(folds[fold_idx]['lpo_test'])
    sub     = pairs.loc[lpo_idx]
    is_new_cell = sub[COL_CELLOSAURUS].isin(cell_ho).to_numpy()
    is_new_drug = sub[COL_DRUG].isin(drug_ho).to_numpy()
    masks = {
        'LPO - nothing new':           (~is_new_cell) & (~is_new_drug),
        'LPO - new cell line only':     is_new_cell   & (~is_new_drug),
        'LPO - new drug only':          (~is_new_cell) & is_new_drug,
        'LPO - fully new (cell+drug)':  is_new_cell   & is_new_drug,
    }
    for name, mask in masks.items():
        print(f'{name}: n={mask.sum():,}')
    return masks

lpo_masks = build_lpo_masks(fold_idx=0)

LPO - nothing new: n=16,599
LPO - new cell line only: n=1,823
LPO - new drug only: n=1,971
LPO - fully new (cell+drug): n=220


## Dataloaders

FP variants share the same `TensorDataset` loaders.
Graph variants get their own `GeoDataLoader` — built separately here so they
can be skipped if running FP variants only.

In [ ]:
# FP variants — shared loaders
fp_train_loader = make_fp_dataloader(train_matrices, BATCH_SIZE, shuffle=True)
fp_val_loader   = make_fp_dataloader(val_matrices,   BATCH_SIZE, shuffle=False)

print('FP loaders ready.')
for rna, protein, drug_fp_t, y in fp_train_loader:
    print(f'  rna={rna.shape} protein={protein.shape} drug_fp={drug_fp_t.shape} y={y.shape}')
    break

In [ ]:
# Graph variants — shared loaders
graph_train_loader = make_graph_dataloader(train_inner_idx, top_cols, BATCH_SIZE, shuffle=True)
graph_val_loader   = make_graph_dataloader(val_idx,          top_cols, BATCH_SIZE, shuffle=False)

graph_lco_loader = make_graph_dataloader(np.array(folds[0]['lco_test']), top_cols, 512, shuffle=False)
graph_ldo_loader = make_graph_dataloader(np.array(folds[0]['ldo_test']), top_cols, 512, shuffle=False)
graph_lto_loader = make_graph_dataloader(np.array(folds[0]['lto_test']), top_cols, 512, shuffle=False)
graph_lpo_loader = make_graph_dataloader(np.array(folds[0]['lpo_test']), top_cols, 512, shuffle=False)

print('Graph loaders ready.')

---
## Variant AX — Drug FP encoder + Drug-Omics cross-attention

`RNA → enc → 256 | Protein → enc → 256 | CrossAttn(q=RNA, kv=Protein) → omics_fused`
`Drug FP → MLP enc → 256 | CrossAttn(q=drug, kv=omics_fused) → fused → predictor`

In [ ]:
DIM_RNA     = train_matrices['rna'].shape[1]
DIM_PROTEIN = train_matrices['protein'].shape[1]
DIM_DRUG_FP = train_matrices['drug_fp'].shape[1]
print(f'RNA dim={DIM_RNA} | Protein dim={DIM_PROTEIN} | Drug FP dim={DIM_DRUG_FP}')

In [ ]:
ax_model = CrossAttentionDRP(
    rna_dim=DIM_RNA,
    protein_dim=DIM_PROTEIN,
    drug_modality='fp',
    drug_integration='attn',
)
print(f'Variant AX parameters: {count_parameters(ax_model):,}')

In [ ]:
ax_model, ax_history = fit_fp_variant(
    model=ax_model,
    train_loader=fp_train_loader,
    val_loader=fp_val_loader,
    variant_name='AX',
    num_epochs=NUM_EPOCHS,
    patience=PATIENCE,
    checkpoint_path=RESULTS_DIR / 'checkpoints' / 'ax_fp_attn.pt',
    device=DEVICE,
)

In [ ]:
ax_preds = {
    'LCO': (lco_matrices['y'], predict_fp_variant(ax_model, lco_matrices)),
    'LDO': (ldo_matrices['y'], predict_fp_variant(ax_model, ldo_matrices)),
    'LTO': (lto_matrices['y'], predict_fp_variant(ax_model, lto_matrices)),
    'LPO': (lpo_matrices['y'], predict_fp_variant(ax_model, lpo_matrices)),
}
ax_results = evaluate_all_splits('AX: fp+attn', ax_preds)
ax_results

---
## Variant AY — Drug FP encoder + concat drug integration

`RNA → enc → 256 | Protein → enc → 256 | CrossAttn(q=RNA, kv=Protein) → omics_fused`
`Drug FP → MLP enc → 256 | concat(omics_fused, drug_emb) → Linear(512,256) → predictor`

In [ ]:
ay_model = CrossAttentionDRP(
    rna_dim=DIM_RNA,
    protein_dim=DIM_PROTEIN,
    drug_modality='fp',
    drug_integration='concat',
)
print(f'Variant AY parameters: {count_parameters(ay_model):,}')

In [ ]:
ay_model, ay_history = fit_fp_variant(
    model=ay_model,
    train_loader=fp_train_loader,
    val_loader=fp_val_loader,
    variant_name='AY',
    num_epochs=NUM_EPOCHS,
    patience=PATIENCE,
    checkpoint_path=RESULTS_DIR / 'checkpoints' / 'ay_fp_concat.pt',
    device=DEVICE,
)

In [ ]:
ay_preds = {
    'LCO': (lco_matrices['y'], predict_fp_variant(ay_model, lco_matrices)),
    'LDO': (ldo_matrices['y'], predict_fp_variant(ay_model, ldo_matrices)),
    'LTO': (lto_matrices['y'], predict_fp_variant(ay_model, lto_matrices)),
    'LPO': (lpo_matrices['y'], predict_fp_variant(ay_model, lpo_matrices)),
}
ay_results = evaluate_all_splits('AY: fp+concat', ay_preds)
ay_results

---
## Variant BX — Drug GNN + Drug-Omics cross-attention

`RNA → enc → 256 | Protein → enc → 256 | CrossAttn(q=RNA, kv=Protein) → omics_fused`
`Drug Graph → GNN → 256 | CrossAttn(q=drug, kv=omics_fused) → fused → predictor`

In [ ]:
bx_model = CrossAttentionDRP(
    rna_dim=DIM_RNA,
    protein_dim=DIM_PROTEIN,
    drug_modality='graph',
    drug_integration='attn',
)
print(f'Variant BX parameters: {count_parameters(bx_model):,}')

In [ ]:
bx_model, bx_history = fit_graph_variant(
    model=bx_model,
    train_loader=graph_train_loader,
    val_loader=graph_val_loader,
    variant_name='BX',
    num_epochs=NUM_EPOCHS,
    patience=PATIENCE,
    checkpoint_path=RESULTS_DIR / 'checkpoints' / 'bx_graph_attn.pt',
    device=DEVICE,
)

In [ ]:
bx_pred_lco, bx_y_lco = predict_graph_variant(bx_model, graph_lco_loader)
bx_pred_ldo, bx_y_ldo = predict_graph_variant(bx_model, graph_ldo_loader)
bx_pred_lto, bx_y_lto = predict_graph_variant(bx_model, graph_lto_loader)
bx_pred_lpo, bx_y_lpo = predict_graph_variant(bx_model, graph_lpo_loader)

bx_preds = {
    'LCO': (bx_y_lco, bx_pred_lco),
    'LDO': (bx_y_ldo, bx_pred_ldo),
    'LTO': (bx_y_lto, bx_pred_lto),
    'LPO': (bx_y_lpo, bx_pred_lpo),
}
bx_results = evaluate_all_splits('BX: graph+attn', bx_preds)
bx_results

---
## Variant BY — Drug GNN + concat drug integration

`RNA → enc → 256 | Protein → enc → 256 | CrossAttn(q=RNA, kv=Protein) → omics_fused`
`Drug Graph → GNN → 256 | concat(omics_fused, drug_emb) → Linear(512,256) → predictor`

In [ ]:
by_model = CrossAttentionDRP(
    rna_dim=DIM_RNA,
    protein_dim=DIM_PROTEIN,
    drug_modality='graph',
    drug_integration='concat',
)
print(f'Variant BY parameters: {count_parameters(by_model):,}')

In [ ]:
by_model, by_history = fit_graph_variant(
    model=by_model,
    train_loader=graph_train_loader,
    val_loader=graph_val_loader,
    variant_name='BY',
    num_epochs=NUM_EPOCHS,
    patience=PATIENCE,
    checkpoint_path=RESULTS_DIR / 'checkpoints' / 'by_graph_concat.pt',
    device=DEVICE,
)

In [ ]:
by_pred_lco, by_y_lco = predict_graph_variant(by_model, graph_lco_loader)
by_pred_ldo, by_y_ldo = predict_graph_variant(by_model, graph_ldo_loader)
by_pred_lto, by_y_lto = predict_graph_variant(by_model, graph_lto_loader)
by_pred_lpo, by_y_lpo = predict_graph_variant(by_model, graph_lpo_loader)

by_preds = {
    'LCO': (by_y_lco, by_pred_lco),
    'LDO': (by_y_ldo, by_pred_ldo),
    'LTO': (by_y_lto, by_pred_lto),
    'LPO': (by_y_lpo, by_pred_lpo),
}
by_results = evaluate_all_splits('BY: graph+concat', by_preds)
by_results

---
## Combined results

In [ ]:
all_results = pd.concat(
    [ax_results, ay_results, bx_results, by_results],
    ignore_index=True,
)

lco_summary = (
    all_results[all_results['split'] == 'LCO']
    [['variant', 'Pearson r', 'Spearman r', 'RMSE', 'R2', 'ROC-AUC']]
    .sort_values('Pearson r', ascending=False)
    .reset_index(drop=True)
)
print('LCO results:')
display(lco_summary)

print('\nPearson r — all splits:')
display(all_results.pivot_table(index='variant', columns='split', values='Pearson r'))

In [ ]:
all_results.to_csv(RESULTS_DIR / 'cross_attention_results_fold0.csv', index=False)
print(f"Saved to {RESULTS_DIR / 'cross_attention_results_fold0.csv'}")

## Learning curves

In [ ]:
def plot_learning_curves(histories: dict, metric: str = 'cor') -> None:
    n = len(histories)
    fig, axes = plt.subplots(1, n, figsize=(4 * n, 4), sharey=True)
    if n == 1:
        axes = [axes]
    ylabel = 'Pearson r' if metric == 'cor' else 'Loss'
    for ax, (name, hist) in zip(axes, histories.items()):
        ax.plot(hist[f'train_{metric}'], label='train')
        ax.plot(hist[f'val_{metric}'],   label='val')
        ax.set_title(name, fontsize=9)
        ax.set_xlabel('epoch')
        ax.set_ylabel(ylabel)
        ax.legend(fontsize=8)
    plt.suptitle(f'Learning curves — {ylabel}', y=1.02)
    plt.tight_layout()
    plt.savefig(RESULTS_DIR / f'learning_curves_{metric}.png', dpi=120, bbox_inches='tight')
    plt.show()

histories = {
    'AX: fp+attn':     ax_history,
    'AY: fp+concat':   ay_history,
    'BX: graph+attn':  bx_history,
    'BY: graph+concat': by_history,
}
plot_learning_curves(histories, metric='cor')
plot_learning_curves(histories, metric='loss')

---
## Multi-fold evaluation — FP variants

AY (fp+concat) and AX (fp+attn) trained and evaluated on all 5 folds.
Feature selection is refit on each fold's train set.
Fold 0 is re-run here for consistency — earlier fold-0 cells remain for reference.

In [22]:
def run_one_fold_fp(fold_idx: int,
                    model_fn,
                    variant_name: str,
                    checkpoint_stem: str) -> pd.DataFrame:
    """Train and evaluate one FP cross-attention variant on one fold.
    model_fn : callable(rna_dim, protein_dim) -> nn.Module
    Returns a tidy DataFrame with a 'fold' column.
    """
    fold = folds[fold_idx]
    train_idx = np.array(fold['train'])

    # Feature selection — fitted on this fold's train set only
    fold_top_cols = {
        'rna':     select_top_variance_omics('rna',     train_idx, TOP_K_FEATURES),
        'protein': select_top_variance_omics('protein', train_idx, TOP_K_FEATURES),
    }

    # Inner train / val split
    inner_idx, val_idx = make_validation_indices(train_idx)

    # Feature matrices
    tr_m  = build_feature_matrix(inner_idx,                  fold_top_cols)
    va_m  = build_feature_matrix(val_idx,                    fold_top_cols)
    lco_m = build_feature_matrix(np.array(fold['lco_test']), fold_top_cols)
    ldo_m = build_feature_matrix(np.array(fold['ldo_test']), fold_top_cols)
    lto_m = build_feature_matrix(np.array(fold['lto_test']), fold_top_cols)
    lpo_m = build_feature_matrix(np.array(fold['lpo_test']), fold_top_cols)

    # LPO masks for this fold
    cell_ho = set(holdout_groups[fold_idx]['cell_lines_held_out'])
    drug_ho = set(holdout_groups[fold_idx]['drugs_held_out'])
    lpo_idx = np.array(fold['lpo_test'])
    sub     = pairs.loc[lpo_idx]
    is_new_cell = sub[COL_CELLOSAURUS].isin(cell_ho).to_numpy()
    is_new_drug = sub[COL_DRUG].isin(drug_ho).to_numpy()
    fold_lpo_masks = {
        'LPO - nothing new':           (~is_new_cell) & (~is_new_drug),
        'LPO - new cell line only':     is_new_cell   & (~is_new_drug),
        'LPO - new drug only':          (~is_new_cell) & is_new_drug,
        'LPO - fully new (cell+drug)':  is_new_cell   & is_new_drug,
    }

    # Dataloaders
    tr_loader = make_fp_dataloader(tr_m, BATCH_SIZE, shuffle=True)
    va_loader = make_fp_dataloader(va_m, BATCH_SIZE, shuffle=False)

    # Train
    rna_dim     = tr_m['rna'].shape[1]
    protein_dim = tr_m['protein'].shape[1]
    model = model_fn(rna_dim, protein_dim)
    ckpt  = RESULTS_DIR / 'checkpoints' / f'{checkpoint_stem}_fold{fold_idx}.pt'
    model, _ = fit_fp_variant(
        model=model, train_loader=tr_loader, val_loader=va_loader,
        variant_name=f'{variant_name} fold {fold_idx}',
        num_epochs=NUM_EPOCHS, patience=PATIENCE,
        checkpoint_path=ckpt, device=DEVICE,
    )

    # Predict
    def pred(m): return predict_fp_variant(model, m)

    # Evaluate with fold-local LPO masks
    rows = []
    for split_name, (y_true, y_pred) in [
        ('LCO', (lco_m['y'], pred(lco_m))),
        ('LDO', (ldo_m['y'], pred(ldo_m))),
        ('LTO', (lto_m['y'], pred(lto_m))),
        ('LPO', (lpo_m['y'], pred(lpo_m))),
    ]:
        if split_name == 'LPO':
            row = {'fold': fold_idx, 'split': 'LPO - all'}
            row.update(evaluate_split(y_true, y_pred))
            rows.append(row)
            for mask_name, mask in fold_lpo_masks.items():
                if mask.sum() == 0:
                    continue
                row = {'fold': fold_idx, 'split': mask_name}
                row.update(evaluate_split(y_true[mask], y_pred[mask]))
                rows.append(row)
        else:
            row = {'fold': fold_idx, 'split': split_name}
            row.update(evaluate_split(y_true, y_pred))
            rows.append(row)

    return pd.DataFrame(rows)


def summarise_folds(df: pd.DataFrame) -> pd.DataFrame:
    return (
        df.groupby('split')['Pearson r']
        .agg(['mean', 'std'])
        .rename(columns={'mean': 'Pearson r mean', 'std': 'Pearson r std'})
        .round(4)
        .reset_index()
    )

### AY — `fp + concat drug integration` — all folds

In [23]:
# AY: fp+concat — all folds
ay_fold_results = []

for fold_idx in range(5):
    print(f'\n--- Fold {fold_idx} | AY: fp+concat ---')
    def make_ay(rna_dim, protein_dim):
        return CrossAttentionDRP(
            rna_dim=rna_dim, protein_dim=protein_dim,
            drug_modality='fp', drug_integration='concat',
        )
    fold_df = run_one_fold_fp(fold_idx, make_ay, 'AY', 'ay_fp_concat')
    ay_fold_results.append(fold_df)

ay_all_folds = pd.concat(ay_fold_results, ignore_index=True)
ay_all_folds.to_csv(RESULTS_DIR / 'ay_fp_concat_all_folds.csv', index=False)
print('\nAY: fp+concat summary:')
display(summarise_folds(ay_all_folds))


--- Fold 0 | AY: fp+concat ---
Training variant: AY fold 0 | params: 3,164,673


epoch 1 val:   0%|          | 0/442 [00:00<?, ?it/s]

  epoch   1 | val loss=3.0714  pearson_r=0.0763


epoch 1 train:   0%|          | 0/1237 [00:00<?, ?it/s]

epoch 2 val:   0%|          | 0/442 [00:00<?, ?it/s]

...

  epoch  17 | val loss=0.9020  pearson_r=0.7485


epoch 17 train:   0%|          | 0/1190 [00:00<?, ?it/s]

epoch 18 val:   0%|          | 0/421 [00:00<?, ?it/s]

  epoch  18 | val loss=0.9132  pearson_r=0.7424
  Early stopping at epoch 18
  Best val loss: 0.8553 -> /content/drive/MyDrive/multiomics_project/results/cross_attention/checkpoints/ay_fp_concat_fold4.pt

AY: fp+concat summary:


,split,Pearson r mean,Pearson r std
0,LCO,0.7954,0.0205
1,LDO,0.3837,0.0605
2,LPO - all,0.8300,0.0188
3,LPO - fully new (cell+drug),0.2858,0.1011
4,LPO - new cell line only,0.8316,0.0033
5,LPO - new drug only,0.4013,0.0707
6,LPO - nothing new,0.8725,0.0075
7,LTO,0.7978,0.0259


### AX — `fp + attention drug integration` — all folds

In [24]:
# AX: fp+attn — all folds
ax_fold_results = []

for fold_idx in range(5):
    print(f'\n--- Fold {fold_idx} | AX: fp+attn ---')
    def make_ax(rna_dim, protein_dim):
        return CrossAttentionDRP(
            rna_dim=rna_dim, protein_dim=protein_dim,
            drug_modality='fp', drug_integration='attn',
        )
    fold_df = run_one_fold_fp(fold_idx, make_ax, 'AX', 'ax_fp_attn')
    ax_fold_results.append(fold_df)

ax_all_folds = pd.concat(ax_fold_results, ignore_index=True)
ax_all_folds.to_csv(RESULTS_DIR / 'ax_fp_attn_all_folds.csv', index=False)
print('\nAX: fp+attn summary:')
display(summarise_folds(ax_all_folds))


--- Fold 0 | AX: fp+attn ---
Training variant: AX fold 0 | params: 3,559,937


epoch 1 val:   0%|          | 0/442 [00:00<?, ?it/s]

  epoch   1 | val loss=3.0122  pearson_r=-0.1071


epoch 1 train:   0%|          | 0/1237 [00:00<?, ?it/s]

epoch 2 val:   0%|          | 0/442 [00:00<?, ?it/s]

...

  epoch  14 | val loss=0.8882  pearson_r=0.7576


epoch 14 train:   0%|          | 0/1190 [00:00<?, ?it/s]

epoch 15 val:   0%|          | 0/421 [00:00<?, ?it/s]

  epoch  15 | val loss=0.9220  pearson_r=0.7472
  Early stopping at epoch 15
  Best val loss: 0.8605 -> /content/drive/MyDrive/multiomics_project/results/cross_attention/checkpoints/ax_fp_attn_fold4.pt

AX: fp+attn summary:


,split,Pearson r mean,Pearson r std
0,LCO,0.7949,0.0180
1,LDO,0.3889,0.0833
2,LPO - all,0.8236,0.0193
3,LPO - fully new (cell+drug),0.2978,0.1770
4,LPO - new cell line only,0.8307,0.0090
5,LPO - new drug only,0.4030,0.0792
6,LPO - nothing new,0.8651,0.0059
7,LTO,0.7972,0.0284


### Combined summary — AX vs AY

In [26]:
# Combined summary — AX vs AY across all folds
ay_all_folds['arm'] = 'AY: fp+concat'
ax_all_folds['arm'] = 'AX: fp+attn'

all_folds_combined = pd.concat([ay_all_folds, ax_all_folds], ignore_index=True)
all_folds_combined.to_csv(RESULTS_DIR / 'fp_variants_all_folds.csv', index=False)


def mean_std_str(s: pd.Series) -> str:
    return f"{s.mean():.3f} ± {s.std():.3f}"
summary = (
    all_folds_combined
    .groupby(['arm', 'split'])[["Pearson r", "Spearman r", "RMSE", "R2", "ROC-AUC"]]
    .agg(mean_std_str)
    .round(4)
    .rename(columns={'mean': 'mean', 'std': 'std'})
)

display(summary)

Pearson r     Spearman r  \
arm           split                                                       
AX: fp+attn   LCO                          0.795 ± 0.018  0.742 ± 0.015   
              LDO                          0.389 ± 0.083  0.353 ± 0.071   
              LPO - all                    0.824 ± 0.019  0.782 ± 0.015   
              LPO - fully new (cell+drug)  0.298 ± 0.177  0.244 ± 0.168   
              LPO - new cell line only     0.831 ± 0.009  0.781 ± 0.012   
              LPO - new drug only          0.403 ± 0.079  0.364 ± 0.079   
              LPO - nothing new            0.865 ± 0.006  0.830 ± 0.009   
              LTO                          0.797 ± 0.028  0.749 ± 0.024   
AY: fp+concat LCO                          0.795 ± 0.021  0.741 ± 0.018   
              LDO                          0.384 ± 0.060  0.348 ± 0.070   
              LPO - all                    0.830 ± 0.019  0.789 ± 0.016   
              LPO - fully new (cell+drug)  0.286 ± 0.101  0.243 ± 0.083   
              LPO - new cell line only     0.832 ± 0.003  0.782 ± 0.014   
              LPO - new drug only          0.401 ± 0.071  0.365 ± 0.086   
              LPO - nothing new            0.873 ± 0.008  0.838 ± 0.010   
              LTO                          0.798 ± 0.026  0.746 ± 0.028   

                                                    RMSE              R2  \
arm           split                                                        
AX: fp+attn   LCO                          1.660 ± 0.067   0.617 ± 0.034   
              LDO                          2.484 ± 0.461   0.032 ± 0.168   
              LPO - all                    1.547 ± 0.089   0.667 ± 0.039   
              LPO - fully new (cell+drug)  2.623 ± 0.425  -0.042 ± 0.187   
              LPO - new cell line only     1.535 ± 0.043   0.676 ± 0.011   
              LPO - new drug only          2.492 ± 0.481   0.045 ± 0.172   
              LPO - nothing new            1.371 ± 0.021   0.741 ± 0.015   
              LTO                          1.667 ± 0.074   0.617 ± 0.051   
AY: fp+concat LCO                          1.646 ± 0.056   0.623 ± 0.031   
              LDO                          2.491 ± 0.430   0.027 ± 0.149   
              LPO - all                    1.512 ± 0.068   0.683 ± 0.032   
              LPO - fully new (cell+drug)  2.638 ± 0.360  -0.049 ± 0.075   
              LPO - new cell line only     1.517 ± 0.031   0.684 ± 0.010   
              LPO - new drug only          2.490 ± 0.431   0.043 ± 0.157   
              LPO - nothing new            1.324 ± 0.022   0.758 ± 0.012   
              LTO                          1.642 ± 0.062   0.629 ± 0.043   

                                                 ROC-AUC  
arm           split                                       
AX: fp+attn   LCO                          0.864 ± 0.008  
              LDO                          0.654 ± 0.037  
              LPO - all                    0.884 ± 0.008  
              LPO - fully new (cell+drug)  0.592 ± 0.075  
              LPO - new cell line only     0.886 ± 0.005  
              LPO - new drug only          0.657 ± 0.050  
              LPO - nothing new            0.908 ± 0.006  
              LTO                          0.867 ± 0.012  
AY: fp+concat LCO                          0.863 ± 0.007  
              LDO                          0.652 ± 0.041  
              LPO - all                    0.887 ± 0.007  
              LPO - fully new (cell+drug)  0.599 ± 0.047  
              LPO - new cell line only     0.886 ± 0.007  
              LPO - new drug only          0.660 ± 0.056  
              LPO - nothing new            0.912 ± 0.006  
              LTO                          0.863 ± 0.014

In [25]:
ay_all_folds

,fold,split,Pearson r,Spearman r,RMSE,R2,ROC-AUC,Slope,Std err
0,0,LCO,0.7942,0.7377,1.6511,0.6195,0.8626,0.9198,0.0053
1,0,LDO,0.4423,0.4080,2.3786,0.1323,0.6846,0.8967,0.0134
2,0,LTO,0.8121,0.7609,1.6232,0.6508,0.8731,0.8989,0.0040
3,0,LPO - all,0.8438,0.8034,1.4655,0.7045,0.8944,0.9480,0.0042
4,0,LPO - nothing new,0.8812,0.8498,1.2981,0.7728,0.9200,0.9581,0.0040
5,0,LPO - new cell line only,0.8282,0.7650,1.5197,0.6772,0.8745,0.9230,0.0146
6,0,LPO - new drug only,0.4529,0.4193,2.3227,0.1207,0.6985,0.9157,0.0406
7,0,LPO - fully new (cell+drug),0.3866,0.3193,2.5729,0.0589,0.6313,0.9518,0.1538
8,1,LCO,0.7967,0.7418,1.6425,0.6268,0.8645,0.8996,0.0052
9,1,LDO,0.3973,0.3744,2.5974,0.0805,0.6578,0.7524,0.0128
